# CT inspection assistant: Colab GPU validation / GPU 实测

This saved copy executes Notebook 05 on Google Colab with a Tesla T4. The final two cells verify CUDA parameter placement, preserved measurements, and three real-model tool workflows. Raw JSON/Markdown artifacts are embedded in the outputs and offered as a ZIP download.

Model: **Qwen/Qwen2.5-1.5B-Instruct**, pinned revision `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`; FP16, no quantization or fine-tuning.

模型选择检测结果与资料之间的对应关系，程序呈现测量数值和带引用的中英文解释。此验证检查 GPU 执行和工具调用，不代表工业检测准确率或制造根因已经验证。

Saved outputs were executed on Colab. Archive metadata has been corrected from the inherited CPU template; the GPU validation manifest records the measured run.

In [1]:
from pathlib import Path
repo = Path('/content/xsim-chip')
if repo.exists():
    !git -C {repo} pull --ff-only
else:
    !git clone -q https://github.com/yuweimin2077-hub/xsim-chip.git {repo}
%cd /content/xsim-chip
%pip install -q -e '.[llm]'


/content/xsim-chip
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.8 MB/s eta 0:00:00
  Building editable for xsim-chip-analysis (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 req

## 1. Configure the question and input / 设置问题和输入

The default is real model inference. Set `RUN_LLM=False` for the explicitly labelled, offline retrieval baseline.
Use `FOUR_BIT=True` only with an available CUDA GPU; the optional installation cell installs bitsandbytes.
Actual GPU availability and memory vary across Colab sessions. A CPU run is supported but slower.


In [2]:
import json
from pathlib import Path
from IPython.display import Markdown, display
from xsim_chip_analysis.assistant import InspectionAssistant, InspectionCase, KnowledgeIndex, render_markdown

ROOT = globals().get('LOCAL_ROOT', Path('/content/xsim-chip'))
OUTPUT = globals().get('LOCAL_OUTPUT', Path('/content/xsim_assistant_outputs'))
CACHE_DIR = globals().get('LOCAL_CACHE', None)
RUN_LLM = not globals().get('LOCAL_OFFLINE', False)
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MODEL_REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'  # tested model revision
FOUR_BIT = False
QUESTION = '检查这个芯片样本的疑似焊料空洞，解释可能的伪影和需要进一步复核的事项。'
LANGUAGE = 'zh'  # or 'en'

# Register trusted paths here. The model cannot choose arbitrary file paths.
# Set DATA to uploaded Notebook 04 outputs to rerun real 3D inspection.
DATA = Path('/content/xsim_volume_outputs')
if (DATA / 'reference_labels.npy').exists() and (DATA / 'reconstructed_attenuation.npy').exists():
    case = InspectionCase('sample', reference_path=DATA / 'reference_labels.npy',
                          reconstruction_path=DATA / 'reconstructed_attenuation.npy', spacing_um=16.0)
else:
    case = InspectionCase('sample', report_path=ROOT / 'docs/results/volume_bridge_2026-10-02/volume_inspection_report.json')
assistant = InspectionAssistant(case)
print('Run LLM:', RUN_LLM, '| Input:', 'saved report' if case.report_path else 'live reconstructed volume')


Run LLM: True | Input: saved report


## 2. Inspect and retrieve / 检测与资料检索

Seven short reviewed source cards use BM25 lexical retrieval, with Chinese query aliases.
They cite the project implementation/results and TI's QFN/SON application note. This is a small
starter corpus, not a complete industrial knowledge base. No document instructions are executed.
The original report retains all components; only representative components are sent to the model.


In [3]:
baseline = assistant.run(QUESTION)
print('Baseline mode:', baseline['mode'])
print('Material-difference components:', baseline['evidence']['component_count'])
for card in baseline['retrieved']:
    print(card['id'], '|', card['title'], '|', card['source_url'])


Baseline mode: retrieval_only
Material-difference components: 82
K-VOID | QFN/SON voiding: reflow and via design | https://www.ti.com/lit/an/slua271c/slua271c.pdf#page=7
K-2D | A slice measures area, not defect volume | https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/xsim_chip_analysis/slice_inspection.py
K-COPPER | Copper differences require independent confirmation | https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/xsim_chip_analysis/inspection.py
K-ARTIFACT | Measured boundary false positives in the synthetic CT baseline | https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/docs/results/volume_bridge_2026-10-02.md


## 3. Run the real model / 运行预训练模型

The model requests allowlisted `inspect_case`, `search_knowledge`, and `finish` actions through a
strict JSON protocol. Invalid actions get at most one repair attempt. `strict=True` makes a failed
model run fail visibly; it will not silently claim model success using a template.

Only retrieved source IDs and real finding IDs can reach the report. Measurements and explanations
are rendered from tool results and reviewed source cards. LoRA/SFT is not performed in this version.


In [4]:
if RUN_LLM:
    if FOUR_BIT:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'bitsandbytes>=0.45,<1'])
    import torch
    if not torch.cuda.is_available():
        torch.set_num_threads(4)
    from xsim_chip_analysis.assistant.huggingface import HuggingFaceBackend
    backend = HuggingFaceBackend(MODEL_ID, revision=MODEL_REVISION,
                                load_in_4bit=FOUR_BIT, cache_dir=CACHE_DIR)
    result = assistant.run(QUESTION, backend, strict=True)
else:
    result = baseline
print('Final mode:', result['mode'])
print('Model:', json.dumps(result['model'], ensure_ascii=False, indent=2))
print('Tool trace:', [(step.get('tool'), step['status']) for step in result['trace']])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Final mode: llm_agent
Model: {
  "backend": "huggingface_transformers",
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "requested_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "resolved_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "device": "cuda:0",
  "load_in_4bit": false,
  "structured_decoding": "lm-format-enforcer; stage-specific JSON schema",
  "max_new_tokens": 384,
  "max_input_tokens": 8192,
  "python": "3.13.15",
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "4.57.6",
    "accelerate": "1.15.0",
    "lm-format-enforcer": "0.11.3"
  }
}
Tool trace: [('inspect_case', 'accepted'), ('search_knowledge', 'accepted'), ('finish', 'accepted')]


## 4. Read and save the report / 查看并保存报告

The full measurement table is retained. Source selections are candidate explanations, not proven
manufacturing causes. The baseline has substantial false positives; adding language does not improve
its segmentation accuracy. No per-joint void percentage, confidence probability or industrial acceptance
decision is invented. See the [annotated CT figure](https://github.com/yuweimin2077-hub/xsim-chip/blob/main/docs/results/volume_bridge_2026-10-02/volume_connection.jpg) from Notebook 04.


In [5]:
OUTPUT.mkdir(parents=True, exist_ok=True)
report_text = render_markdown(result, LANGUAGE)
(OUTPUT / 'report.md').write_text(report_text, encoding='utf-8')
(OUTPUT / 'assistant_result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT / 'retrieval_baseline.json').write_text(json.dumps(baseline, ensure_ascii=False, indent=2), encoding='utf-8')
display(Markdown(report_text))
print('Saved to:', OUTPUT)


# CT 缺陷筛查报告

模式: `llm_agent`

本报告中的数值来自检测工具。模型选择资料与区域的对应关系，解释文字来自经过审核的资料卡。

## 检测事实

- 3D; shape: [16, 129, 129]; spacing: 16.0 um.
- Candidate provenance: `fixed_threshold_segmentation_of_reconstructed_slice_stack`.
- 共 82 个材料差异连通域；这不是已确认缺陷的数量。

| 编号 | 筛查信号 | 大小 | 中心坐标 (um) |
| --- | --- | --- | --- |
| D0001 | 疑似多余铜 | 8126464.000 um3 | (122.153, 1471.887, 1245.613) |
| D0002 | 疑似多余铜 | 3784704.000 um3 | (119.775, 1471.758, 576.727) |
| D0003 | 疑似焊料缺失/断开 | 2478080.000 um3 | (119.802, 1471.524, 577.269) |
| D0004 | 疑似焊料缺失/断开 | 2375680.000 um3 | (116.386, 1471.503, 1019.366) |
| D0005 | 疑似焊料缺失/断开 | 2351104.000 um3 | (116.739, 1471.666, 1476.181) |
| D0006 | 疑似多余焊料/桥连 | 1425408.000 um3 | (168.000, 1480.276, 1248.000) |
| D0007 | 疑似铜缺失/断开 | 1089536.000 um3 | (87.940, 577.023, 1016.662) |
| D0008 | 疑似焊料缺失/断开 | 692224.000 um3 | (120.331, 1472.284, 1024.000) |
| D0009 | 疑似介质缺失 | 3190784.000 um3 | (152.154, 1476.929, 1248.144) |
| D0010 | 疑似介质多余/错位 | 1089536.000 um3 | (87.940, 577.023, 1016.662) |
| D0011 | 疑似多余铜 | 397312.000 um3 | (120.577, 1472.495, 1024.000) |
| D0012 | 疑似介质多余/错位 | 266240.000 um3 | (120.615, 1472.000, 1024.000) |
| D0013 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1392.000, 496.000) |
| D0014 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1392.000, 944.000) |
| D0015 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1392.000, 1552.000) |
| D0016 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1552.000, 496.000) |
| D0017 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1552.000, 656.000) |
| D0018 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1552.000, 944.000) |
| D0019 | 疑似介质缺失 | 196608.000 um3 | (120.000, 1552.000, 1552.000) |
| D0020 | 疑似介质缺失 | 192512.000 um3 | (118.128, 1391.660, 655.660) |
| D0021 | 疑似介质缺失 | 94208.000 um3 | (125.217, 1360.000, 1003.130) |
| D0022 | 疑似介质缺失 | 73728.000 um3 | (121.778, 1360.000, 1041.778) |
| D0023 | 疑似介质缺失 | 73728.000 um3 | (127.111, 1584.000, 1006.222) |
| D0024 | 疑似介质缺失 | 69632.000 um3 | (114.824, 1455.059, 912.000) |
| D0025 | 疑似介质缺失 | 69632.000 um3 | (113.882, 1455.059, 1584.000) |
| D0026 | 疑似介质缺失 | 69632.000 um3 | (113.882, 1488.941, 912.000) |
| D0027 | 疑似介质缺失 | 69632.000 um3 | (114.824, 1584.000, 1040.941) |
| D0028 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 352.000) |
| D0029 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 416.000) |
| D0030 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 800.000) |
| D0031 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1008.000) |
| D0032 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1072.000) |
| D0033 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1184.000) |
| D0034 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1248.000) |
| D0035 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1472.000) |
| D0036 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1040.000, 1632.000) |
| D0037 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1360.000, 560.000) |
| D0038 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1360.000, 592.000) |
| D0039 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1360.000, 1456.000) |
| D0040 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1360.000, 1488.000) |
| D0041 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1456.000, 464.000) |
| D0042 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1456.000, 688.000) |
| D0043 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1488.000, 464.000) |
| D0044 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1488.000, 688.000) |
| D0045 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1488.000, 1584.000) |
| D0046 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1584.000, 560.000) |
| D0047 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1584.000, 592.000) |
| D0048 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1584.000, 1456.000) |
| D0049 | 疑似介质缺失 | 65536.000 um3 | (120.000, 1584.000, 1488.000) |
| D0050 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 352.000) |
| D0051 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 416.000) |
| D0052 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 800.000) |
| D0053 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1008.000) |
| D0054 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1072.000) |
| D0055 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1184.000) |
| D0056 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1248.000) |
| D0057 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1472.000) |
| D0058 | 疑似多余铜 | 65536.000 um3 | (120.000, 1040.000, 1632.000) |
| D0059 | 疑似介质缺失 | 61440.000 um3 | (128.000, 1712.000, 160.000) |
| D0060 | 疑似介质缺失 | 53248.000 um3 | (96.000, 1040.000, 592.000) |
| D0061 | 疑似介质缺失 | 53248.000 um3 | (144.000, 784.000, 1248.000) |
| D0062 | 疑似介质缺失 | 53248.000 um3 | (144.000, 1040.000, 864.000) |
| D0063 | 疑似多余铜 | 53248.000 um3 | (96.000, 1040.000, 592.000) |
| D0064 | 疑似多余铜 | 53248.000 um3 | (144.000, 784.000, 1248.000) |
| D0065 | 疑似多余铜 | 53248.000 um3 | (144.000, 1040.000, 864.000) |
| D0066 | 疑似介质缺失 | 49152.000 um3 | (152.000, 608.000, 800.000) |
| D0067 | 疑似多余铜 | 49152.000 um3 | (152.000, 608.000, 800.000) |
| D0068 | 疑似介质缺失 | 40960.000 um3 | (72.000, 848.000, 1248.000) |
| D0069 | 疑似多余铜 | 40960.000 um3 | (72.000, 848.000, 1248.000) |
| D0070 | 疑似介质缺失 | 36864.000 um3 | (64.000, 784.000, 800.000) |
| D0071 | 疑似介质缺失 | 36864.000 um3 | (64.000, 848.000, 1072.000) |
| D0072 | 疑似介质缺失 | 36864.000 um3 | (128.000, 848.000, 592.000) |
| D0073 | 疑似介质多余/错位 | 36864.000 um3 | (176.000, 1088.000, 1840.000) |
| D0074 | 疑似铜缺失/断开 | 36864.000 um3 | (176.000, 1088.000, 1840.000) |
| D0075 | 疑似多余铜 | 36864.000 um3 | (64.000, 784.000, 800.000) |
| D0076 | 疑似多余铜 | 36864.000 um3 | (64.000, 848.000, 1072.000) |
| D0077 | 疑似多余铜 | 36864.000 um3 | (128.000, 848.000, 592.000) |
| D0078 | 疑似介质缺失 | 32768.000 um3 | (56.000, 1040.000, 1696.000) |
| D0079 | 疑似介质缺失 | 32768.000 um3 | (184.000, 784.000, 1008.000) |
| D0080 | 疑似介质缺失 | 32768.000 um3 | (184.000, 1712.000, 1872.000) |
| D0081 | 疑似多余铜 | 32768.000 um3 | (56.000, 1040.000, 1696.000) |
| D0082 | 疑似多余铜 | 32768.000 um3 | (184.000, 784.000, 1008.000) |

坐标顺序：z, y, x

## 候选解释与复核

解释只覆盖选出的代表性区域，不代表已逐一复核全部连通域。

- **D0001** — 项目通过比较材料标签给出铜差异信号。这些标签并未测量电气连续性，也不能确认电镀或蚀刻根因。 [K-COPPER](https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/xsim_chip_analysis/inspection.py)
  先复核重建和分割；需要确认物理原因时，再结合导通测试或工艺记录。
- **D0009, D0007, D0010** — 已归档的逐层实验显示，Hann FBP 与固定阈值分割会导致边界误分，包括在无缺陷层中预测出焊料缺失。候选差异不等于已确认的物理缺陷。 [K-ARTIFACT](https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/docs/results/volume_bridge_2026-10-02.md)
  同时检查灰度图、参考体对齐情况和相邻切片；改进分割后，用独立样本验证。
- **D0003** — TI 指出，回流过程中的助焊剂活性和散热过孔处理会影响 QFN/SON 空洞。这只能提供候选解释，不能证明本合成样本的制造根因。 [K-VOID](https://www.ti.com/lit/an/slua271c/slua271c.pdf#page=7)
  若实际封装适用，检查焊膏供应商的回流曲线和过孔设计，并结合相邻切片复核空洞。
- **D0006** — TI 将焊桥预防与焊盘几何及间距联系起来。多余焊料差异属于筛查信号，成像误差也可能产生这种信号。 [K-BRIDGE](https://www.ti.com/lit/an/slua271c/slua271c.pdf#page=6)
  检查实际焊盘布局并复核重建区域的连通性，不把特定封装的验收限值直接套用到本演示。

## 需要保留的限制

- 固定阈值与成像伪影可能造成误报或漏检；请复核原始灰度图及相邻切片。
- 未确认制造根因，没有校准后的置信概率或工业合格判定。
- 未计算焊点体积占比：需要明确的焊点分割与分母；二维面积不能换称三维体积。

## 资料来源

- [K-COPPER: Copper differences require independent confirmation](https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/xsim_chip_analysis/inspection.py) — _DEFECT_KINDS and analyze_against_reference. Project screening logic, not externally validated causal evidence.
- [K-VOID: QFN/SON voiding: reflow and via design](https://www.ti.com/lit/an/slua271c/slua271c.pdf#page=7) — SLUA271C, December 2023, section 3.4.1, printed page 7. QFN/SON assembly guidance; applicability to other package geometries requires review.
- [K-BRIDGE: QFN/SON solder bridging and land geometry](https://www.ti.com/lit/an/slua271c/slua271c.pdf#page=6) — SLUA271C, December 2023, sections 3.2-3.4, printed pages 6-7. QFN/SON geometry guidance; no universal pass/fail rule.
- [K-ARTIFACT: Measured boundary false positives in the synthetic CT baseline](https://github.com/yuweimin2077-hub/xsim-chip/blob/cfda8bf/docs/results/volume_bridge_2026-10-02.md) — Measured result and CUDA per-signature evaluation. Project synthetic baseline; not a general scanner calibration.


Saved to: /content/xsim_assistant_outputs


## Next questions / 后续问题

Change `QUESTION` and rerun the baseline/model cells; the loaded model can be reused.
For 2D, register a single-slice report or aligned reference/reconstruction arrays.
For your own files, upload them using Colab's Files panel and explicitly register their paths above;
raw projection images are not accepted by the inspection tool. Use Notebook 04 to reconstruct first.

Before disconnecting, download the output directory or copy it to persistent storage. The Colab VM is temporary.
The source code and notebook are on GitHub; model weights and personal inputs are not committed.


In [6]:
# Validate the actual Colab CUDA run and archive auditable results.
import json, platform, subprocess, time, zipfile
from datetime import datetime, timezone
import google.colab
import torch

assert torch.cuda.is_available(), 'GPU validation requires a CUDA runtime'
assert result['mode'] == 'llm_agent', 'The notebook model run must succeed'
assert result['evidence'] == baseline['evidence'], 'Measurements must stay unchanged'
parameter_devices = sorted({str(p.device) for p in backend.model.parameters()})
assert all(d.startswith('cuda') for d in parameter_devices), parameter_devices
validation = {
    'executed_utc': datetime.now(timezone.utc).isoformat(),
    'environment': 'Google Colab', 'colab_run': True,
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
    'gpu_name': torch.cuda.get_device_name(0),
    'gpu_total_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30,
    'cuda_version': torch.version.cuda, 'python': platform.python_version(),
    'parameter_devices': parameter_devices,
    'parameter_dtypes': sorted({str(p.dtype) for p in backend.model.parameters()}),
    'model': result['model'],
    'notebook_elapsed_seconds': result['elapsed_seconds'],
    'notebook_trace': [{'tool': s.get('tool'), 'status': s['status']} for s in result['trace']],
    'measurements_equal_to_baseline': result['evidence'] == baseline['evidence'],
    'scope': 'GPU execution and grounding smoke tests, not industrial accuracy validation',
    'cases': [],
}
print('Verified Colab GPU:', validation['gpu_name'], parameter_devices, flush=True)
torch.cuda.reset_peak_memory_stats()
for name, relative, question in [
    ('volume_void', 'volume_bridge_2026-10-02/volume_inspection_report.json', '检查疑似焊料空洞，是否可能是伪影？'),
    ('slice_area', 'slice_closed_loop_2026-10-01/inspection_report.json', 'Can this single slice establish a 3D void volume fraction?'),
    ('volume_copper', 'volume_bridge_2026-10-02/volume_inspection_report.json', 'Explain copper open screening and what evidence is still missing.'),
]:
    torch.cuda.synchronize()
    start = time.perf_counter()
    checked = InspectionAssistant(InspectionCase(name, report_path=ROOT / 'docs/results' / relative)).run(question, backend, strict=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    assert checked['mode'] == 'llm_agent'
    (OUTPUT / (name + '.json')).write_text(json.dumps(checked, ensure_ascii=False, indent=2), encoding='utf-8')
    (OUTPUT / (name + '.md')).write_text(render_markdown(checked), encoding='utf-8')
    row = {'case': name, 'mode': checked['mode'], 'dimensions': checked['evidence']['dimensions'],
           'components': checked['evidence']['component_count'], 'elapsed_seconds': elapsed,
           'accepted_steps': sum(s['status'] == 'accepted' for s in checked['trace']),
           'rejected_steps': sum(s['status'] == 'rejected' for s in checked['trace'])}
    validation['cases'].append(row)
    print(json.dumps(row), flush=True)
validation['peak_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
validation['peak_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
(OUTPUT / 'gpu_validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT / 'report_en.md').write_text(render_markdown(result, 'en'), encoding='utf-8')
archive_path = OUTPUT.parent / 'xsim_colab_gpu_validation.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(OUTPUT.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.md'}:
            archive.write(file, file.name)
print('GPU VALIDATION PASSED:', json.dumps(validation, ensure_ascii=False, indent=2))
print('Download archive:', archive_path)

from IPython.display import JSON
artifact_bundle = {p.name: p.read_text(encoding='utf-8') for p in sorted(OUTPUT.iterdir()) if p.is_file() and p.suffix in {'.json', '.md'}}
display(JSON(artifact_bundle, expanded=False))


Verified Colab GPU: Tesla T4 ['cuda:0']
{"case": "volume_void", "mode": "llm_agent", "dimensions": 3, "components": 82, "elapsed_seconds": 80.239305263, "accepted_steps": 3, "rejected_steps": 0}
{"case": "slice_area", "mode": "llm_agent", "dimensions": 2, "components": 6, "elapsed_seconds": 39.87589474299966, "accepted_steps": 3, "rejected_steps": 0}
{"case": "volume_copper", "mode": "llm_agent", "dimensions": 3, "components": 82, "elapsed_seconds": 38.9836566189997, "accepted_steps": 3, "rejected_steps": 0}
GPU VALIDATION PASSED: {
  "executed_utc": "2026-10-03T23:30:57.190973+00:00",
  "environment": "Google Colab",
  "colab_run": true,
  "git_commit": "efd65eba69ac40167dd21612b159a618280aa644",
  "gpu_name": "Tesla T4",
  "gpu_total_memory_gib": 14.56317138671875,
  "cuda_version": "13.0",
  "python": "3.13.15",
  "parameter_devices": [
    "cuda:0"
  ],
  "parameter_dtypes": [
    "torch.float16"
  ],
  "model": {
    "backend": "huggingface_transformers",
    "model_id": "Qwen/Qwe

<IPython.core.display.JSON object>

In [7]:
from IPython.display import HTML
import base64
summary = f"""## Colab GPU validation: PASSED

Model: **{validation['model']['model_id']}**
GPU: **{validation['gpu_name']}** | Parameters: **{', '.join(validation['parameter_devices'])}** | **FP16**
Measured peak allocated: **{validation['peak_allocated_gib']:.2f} GiB**; peak reserved: **{validation['peak_reserved_gib']:.2f} GiB**
Measurements unchanged: **{validation['measurements_equal_to_baseline']}**

| Case | Seconds | Accepted actions | Rejected actions |
| --- | ---: | ---: | ---: |
"""
for row in validation['cases']:
    summary += f"| {row['case']} | {row['elapsed_seconds']:.2f} | {row['accepted_steps']} | {row['rejected_steps']} |\n"
summary += "\nGPU execution and grounding smoke tests; no LoRA/SFT or industrial accuracy claim."
display(Markdown(summary))
encoded_archive = base64.b64encode(archive_path.read_bytes()).decode('ascii')
display(HTML('<a download="xsim_colab_gpu_validation.zip" href="data:application/zip;base64,' + encoded_archive + '">Download GPU validation archive</a>'))


## Colab GPU validation: PASSED

Model: **Qwen/Qwen2.5-1.5B-Instruct**  
GPU: **Tesla T4** | Parameters: **cuda:0** | **FP16**  
Measured peak allocated: **3.80 GiB**; peak reserved: **4.63 GiB**  
Measurements unchanged: **True**

| Case | Seconds | Accepted actions | Rejected actions |
| --- | ---: | ---: | ---: |
| volume_void | 80.24 | 3 | 0 |
| slice_area | 39.88 | 3 | 0 |
| volume_copper | 38.98 | 3 | 0 |

GPU execution and grounding smoke tests; no LoRA/SFT or industrial accuracy claim.